In [ ]:
# HALCINATION PROCESS IS PROCESSING THE DATA BASED ON CONTEXT MEMORY

In [36]:
!pip install chromadb sentence-transformers -q

In [37]:
!pip install sentence-transformers chromadb groq pandas -q
print("All the libraries Installed Successfully")

All the libraries Installed Successfully


In [39]:
!pip install -U chromadb
!pip install -U opentelemetry-api opentelemetry-sdk
!pip install -U opentelemetry-exporter-otlp

  Using cached opentelemetry_api-1.42.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached opentelemetry_semantic_conventions-0.63b1-py3-none-any.whl.metadata (2.4 kB)
Using cached opentelemetry_api-1.42.1-py3-none-any.whl (61 kB)
Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl (170 kB)
Using cached opentelemetry_semantic_conventions-0.63b1-py3-none-any.whl (203 kB)
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.22.0
    Uninstalling opentelemetry-api-1.22.0:
      Successfully uninstalled opentelemetry-api-1.22.0
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.43b0
    Uninstalling opentelemetry-semantic-conventions-0.43b0:
      Successfully uninstalled opentelemetry-semantic-conventions-0.43b0
  Attempting uninstall: opentelemetry-sdk
    Found existing installation: opentelem

  Using cached opentelemetry_exporter_otlp_proto_grpc-1.42.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_exporter_otlp_proto_common-1.42.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached opentelemetry_proto-1.42.1-py3-none-any.whl.metadata (2.3 kB)
Using cached opentelemetry_exporter_otlp_proto_grpc-1.42.1-py3-none-any.whl (19 kB)
Using cached opentelemetry_exporter_otlp_proto_common-1.42.1-py3-none-any.whl (17 kB)
Using cached opentelemetry_proto-1.42.1-py3-none-any.whl (71 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 10.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.9
    Uninstalling protobuf-4.25.9:
      Successfully uninstalled protobuf-4.25.9
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.22.0
    Uninstalling opentelemetry-proto-1.22.0:
      Successfully uninstalled opentelemetry-proto-1.22.0
  Attempting uninstall: opentelemetry-expor

In [1]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
print("All Packages are Installed Successfully")

All Packages are Installed Successfully


In [8]:
GROQ_API_KEY = "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"
# OS environment is used to hide the API key which act as locker that holds API and make only the neccessary API calls
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)
print("Groq API client initialized.")
print("Note: If you see an authentication error later, double click your API key.")

Groq API client initialized.
Note: If you see an authentication error later, double click your API key.


In [71]:
df = pd.read_csv("/content/drive/MyDrive/Summer_Internship_2026/college_notes.csv")
print("=== Data Loaded Successfully ===")
print()
print("Shape the DataSet : ",df.shape)
print()
print("Column Names : ",df.columns.to_list())
print()
print("First 3 Rows : ",df.head(3))

=== Data Loaded Successfully ===

Shape the DataSet :  (15, 4)

Column Names :  ['note_id', 'subject', 'topic', 'content']

First 3 Rows :    note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  


In [72]:
print("Subject in the datasets : ")
print(df['subject'].value_counts())
print("-"*40)
print("Sample of topics : ")
print(df[['note_id','subject','topic']].to_string(index=False))
print("-"*40)
print("Length of content (number of characters) : for each note : ")
df['content_length'] = df['content'].apply(len)
print(df[['note_id','content_length']].to_string(index=False))

Subject in the datasets : 
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64
----------------------------------------
Sample of topics : 
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI R

In [73]:
documents = df['content'].to_list()

ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas = [
    {"Subject": row['subject'],"topic":row['topic']}
    for row in df.to_dict('records')
]
print(f"total chunks prepared : {len(documents)}")
print(f"First document ID :{ids[0]}")
print(f"First MetaData : {metadatas[0]}")
print(f"First 100 characters of doc : {documents[0][:100]}....")

total chunks prepared : 15
First document ID :note_N001
First MetaData : {'Subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 characters of doc : ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc....


In [74]:
from sentence_transformers import SentenceTransformer

print("Loading Embedding model....")
print("This may 30-60 seconds on first run - model is being doenloaded")
print("Subsequent runs will be faster as the model is cached")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded successfully")
test_embedding = embedding_model.encode("This is a sentence")
print("Test embedding shape:", test_embedding.shape)
print("First 5 values of test embedding:", test_embedding[:5])

Loading Embedding model....
This may 30-60 seconds on first run - model is being doenloaded
Subsequent runs will be faster as the model is cached


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded successfully
Test embedding shape: (384,)
First 5 values of test embedding: [ 0.05048309  0.088006    0.00487488  0.03626884 -0.00101813]


In [75]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="college_notes_rag")
print("chromadb client created")
print("Collection name: college_notes_rag")
print("Documents in collection so far:", collection.count())

chromadb client created
Collection name: college_notes_rag
Documents in collection so far: 15


In [76]:
print("Generating embeddings for all 15 notes ...")
print("This may take 15 - 30 seconds...")
embeddings = embedding_model.encode(documents, show_progress_bar=True)
print(f"Embedding matrix shape : {embeddings.shape}")
embeddings_list = embeddings.tolist()
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
    embeddings=embeddings_list
)
print(f"Document successfully added to chromaDB.")
print(f"Total documents in collection : {collection.count()}")

Generating embeddings for all 15 notes ...
This may take 15 - 30 seconds...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape : (15, 384)
Document successfully added to chromaDB.
Total documents in collection : 15


In [1]:
# CHROMADB RETRIEVAL - FINDING RELEVANT CHUNKS
# USES TO ASK QUESTION
# QUESTION WILL BE CONVERTED INTO VECTOR
# CHROMADB WILL COMPARE THE QUERY VECTOR TO ALL THE EXISTING VECTORS
# RETURN THE TOP K MOST SIMILAR DOCUMENT
# SORTED MAINLY BASED ON THE DISTANCE

# CONTEXT INJECTION
# WE ADD THE RETRIEVED DOCUMENT CHUNKS DIRECTLY INTO LLM PROMPT BEFORE THE USER ASKS QUESTIONS
# THE LLM SHOULD ANSWER ONLY BASED ON THE CONTEXTED DOCUMENT

# MAKE A RAG PROMPT - RETRIEVAL AUGMENTED GENERATION
# THE USER MESSAGE AND THE SYSTEM MESSAGE ARE THE CONTEXT INJECTION FOR THE CONSCISE OUTCOME

print("=== READ THE NOTES BEFORE HANDS - ON ===")

=== READ THE NOTES BEFORE HANDS - ON ===


In [78]:
def retrieve_relevant_chunks(question, top_k=3):

    # Convert question into embedding
    question_embedding = embedding_model.encode(
        question
    ).tolist()

    # Search in ChromaDB
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )

    return results


print("Retrieval function created successfully")

print(
    "Function: retrieve_relevant_chunks(question, top_k=3)"
)

Retrieval function created successfully
Function: retrieve_relevant_chunks(question, top_k=3)


In [79]:
test_question = "What is ETL and how does it work in data engineering?"
print("Test question:", test_question)

results = retrieve_relevant_chunks(test_question,top_k=3)
print("Top 3 retrieved chunks:")

for i,(doc,dist,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
   print("Result",i+1)
   print("Subject:", meta['Sunject'])
   print("Topic:", meta['topic'])
   print(f"Distance: {dist:.4f}")
   print("Content:", doc[:120])

Test question: What is ETL and how does it work in data engineering?
Top 3 retrieved chunks:
Result 1
Subject: Data Engineering
Topic: ETL Pipelines
Distance: 0.2269
Content: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i
Result 2
Subject: Data Engineering
Topic: APIs and Data Collection
Distance: 1.0690
Content: An API or Application Programming Interface allows two software applications to talk to each other. In data engineering 
Result 3
Subject: Python Programming
Topic: Data Visualization
Distance: 1.3375
Content: Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo


In [70]:
def build_context_from_result(results):
  """
  """
  context_parts = []
  for i, (doc,meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):
    # Fixed indentation and changed 'subject' to 'Sunject'
    chunk_text = f"[Source {i+1} : {meta['Sunject']} - {meta['topic']}]"
    context_parts.append(chunk_text + doc)
  return "\n\n" + "\n---\n".join(context_parts)

# Added to print the output
context_output = build_context_from_result(results)
print(context_output)




[Source 1 : Data Engineering - ETL Pipelines]ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.
---
[Source 2 : Data Engineering - APIs and Data Collection]An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.
---
[Source 3 : Python Programming - Data Visualization]Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplotlib and Seaborn are used to create bar charts line plots histograms and pie charts that help humans understand patterns in data.


In [13]:
def generate_rag_answer(question,context):
  """
  """
  system_prompt = """ You are helpful Path Navigator to avoid traffic issue.
  You will be able to perform like a google navigator and helpful for efficient path finding

  Path Navigator System Prompt Rules :

  1. Act as a helpful Path Navigator for route planning and navigation.
  2. Provide efficient path-finding assistance similar to a navigation system.
  3. Prioritize routes that avoid traffic congestion, roadblocks, accidents, and delays.
  4. Recommend the fastest and most practical route to the destination.
  5. Suggest alternative routes when the primary route is congested or unavailable.
  6. Include estimated travel time and distance whenever available.
  7. Consider factors such as traffic conditions, road closures, toll roads, and travel restrictions.
  8. Clearly explain why a particular route is recommended.
  9. Compare multiple route options when more than one viable path exists.
  10. Prioritize user safety and legal road usage in all recommendations.
  11. Provide clear, concise, and easy-to-follow navigation guidance.
  12. Ask for missing information (such as origin or destination) when necessary.
  13. Update recommendations when new traffic or route information becomes available.
  14. Avoid recommending unsafe, illegal, or restricted roads.
  15. Focus on minimizing travel time while maintaining route reliability.
  """

  use_prompt = f""" Context from Knowledge Base :
  {context}

  Drivers Question : {question}
  Please answer the question based on the context provided above """
  response = groq_client.chat.completions.create(
      model="llama-3.1-8b-instant",
      messages =[
          {"role":"system","content":system_prompt},
          {"role":"user","content":use_prompt}
      ],
      temperature = 0.1,
      max_tokens = 500 # Corrected parameter name from max_token to max_tokens
  )
  answer = response.choices[0].message.content
  return answer
print("=== RAG generation function is defined ===")


=== RAG generation function is defined ===
